# SafeCityAI – Traffic Violation Detection using YOLOv5
## Capstone Project Training Notebook (Roboflow Dataset Integration)

This notebook provides a complete end-to-end pipeline for training a custom **YOLOv5** model on the real **Roboflow Traffic Dataset** for traffic violation enforcement.

### Discovered Dataset Class Mapping (Roboflow Order):
- `0`: **Helmet** (Rider wearing protective helmet)
- `1`: **LicensePlate** (Vehicle license plate for identification)
- `2`: **NoHelmet** (Rider operating motorcycle without helmet - violation)

---

## Step 1: Check GPU Availability

In [ ]:
# Check CUDA GPU availability in Google Colab
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select GPU (e.g. T4 GPU).")

!nvidia-smi

## Step 2: Clone YOLOv5 Repository & Install Dependencies

In [ ]:
import os
# Clone YOLOv5 official repository
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git

%cd yolov5
!pip install -qr requirements.txt

import utils
display = utils.notebook_init()

## Step 3: Mount Google Drive & Dataset Ingestion Options

Option A: Mount Google Drive where `roboflow_dataset.zip` is stored.
Option B: Directly upload `roboflow_dataset.zip` via Google Colab files sidebar.

In [ ]:
# Mount Google Drive to access stored dataset zip or export trained best.pt
from google.colab import drive
drive.mount('/content/drive')

## Step 4: Extract Custom Roboflow Dataset

In [ ]:
# Path to dataset zip on Drive or uploaded to Colab root
DRIVE_ZIP_PATH = '/content/drive/MyDrive/roboflow_dataset.zip'
LOCAL_ZIP_PATH = '/content/roboflow_dataset.zip'
EXTRACT_PATH = '/content/dataset'

if os.path.exists(DRIVE_ZIP_PATH):
    print(f"Extracting dataset from Google Drive: {DRIVE_ZIP_PATH}...")
    !unzip -q "{DRIVE_ZIP_PATH}" -d "{EXTRACT_PATH}"
elif os.path.exists(LOCAL_ZIP_PATH):
    print(f"Extracting uploaded dataset: {LOCAL_ZIP_PATH}...")
    !unzip -q "{LOCAL_ZIP_PATH}" -d "{EXTRACT_PATH}"
else:
    print("WARNING: Dataset zip not found at Drive or Colab root!")
    print("Please compress your dataset/roboflow_dataset folder to 'roboflow_dataset.zip' and upload it.")

## Step 5: Comprehensive Dataset Verification & Validation

In [ ]:
import glob

def verify_split_files(dataset_root):
    splits = ['train', 'valid', 'test']
    counts = {}
    for s in splits:
        img_pattern = os.path.join(dataset_root, s, 'images', '*.*')
        lbl_pattern = os.path.join(dataset_root, s, 'labels', '*.txt')
        imgs = glob.glob(img_pattern)
        lbls = glob.glob(lbl_pattern)
        counts[s] = {'images': len(imgs), 'labels': len(lbls)}
    return counts

# Handle nested folder structure if unzipped into roboflow_dataset
DATASET_ROOT = EXTRACT_PATH
if os.path.exists(os.path.join(EXTRACT_PATH, 'roboflow_dataset')):
    DATASET_ROOT = os.path.join(EXTRACT_PATH, 'roboflow_dataset')

counts = verify_split_files(DATASET_ROOT)

print("=== SafeCityAI Real Dataset Verification ===")
print(f"Dataset Root Path: {DATASET_ROOT}")
print(f"Number of Classes (nc): 3")
print(f"Discovered Class Mapping: {{0: 'Helmet', 1: 'LicensePlate', 2: 'NoHelmet'}}")
print(f"----------------------------------------------")
print(f"Train Split: {counts['train']['images']} images, {counts['train']['labels']} labels")
print(f"Valid Split: {counts['valid']['images']} images, {counts['valid']['labels']} labels")
print(f"Test Split:  {counts['test']['images']} images, {counts['test']['labels']} labels")
print(f"----------------------------------------------")
assert counts['train']['images'] > 0, "Train images missing! Please verify dataset zip extraction."
assert counts['valid']['images'] > 0, "Validation images missing! Please verify dataset zip extraction."

## Step 6: Display Sample Bounding Box Annotations

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

def preview_sample_annotation(dataset_root):
    train_imgs = glob.glob(os.path.join(dataset_root, 'train', 'images', '*.*'))
    if not train_imgs:
        print("No training images found to visualize.")
        return
    
    img_path = random.choice(train_imgs)
    img = cv2.imread(img_path)
    if img is None:
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    lbl_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'
    classes_map = {0: 'Helmet', 1: 'LicensePlate', 2: 'NoHelmet'}
    colors = {0: (0, 255, 0), 1: (255, 165, 0), 2: (255, 0, 0)}
    
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if not parts: continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:])
            xmin = int((xc - bw/2) * w)
            ymin = int((yc - bh/2) * h)
            xmax = int((xc + bw/2) * w)
            ymax = int((yc + bh/2) * h)
            
            color = colors.get(cls_id, (255, 255, 255))
            cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, 2)
            cv2.putText(img, classes_map.get(cls_id, str(cls_id)), (xmin, max(ymin-5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Sample: {os.path.basename(img_path)}")
    plt.show()

preview_sample_annotation(DATASET_ROOT)

## Step 7: Create Dataset Configuration (`data.yaml`)

In [ ]:
data_yaml_content = f"""
path: {DATASET_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 3
names:
  0: Helmet
  1: LicensePlate
  2: NoHelmet
"""

data_yaml_path = '/content/data.yaml'
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content.strip())

print(f"Configured dataset definition at {data_yaml_path}:")
!cat {data_yaml_path}

## Step 8: Train YOLOv5 Model using Transfer Learning

In [ ]:
# Train YOLOv5s model with transfer learning on 3 classes
# Hyperparameters: img_size=640, batch_size=16, epochs=50
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data /content/data.yaml \
    --weights yolov5s.pt \
    --name safecityai_yolov5_run \
    --cache

## Step 9: Locate Best Trained Model Weights (`best.pt`)

In [ ]:
BEST_WEIGHTS_PATH = 'runs/train/safecityai_yolov5_run/weights/best.pt'

if os.path.exists(BEST_WEIGHTS_PATH):
    print(f"SUCCESS! Best model weights saved at: {BEST_WEIGHTS_PATH}")
else:
    print(f"Weights file not found at {BEST_WEIGHTS_PATH}. Check training log above for errors.")

## Step 10: Display Training Performance Plots

In [ ]:
from IPython.display import Image, display

results_png = 'runs/train/safecityai_yolov5_run/results.png'
confusion_matrix_png = 'runs/train/safecityai_yolov5_run/confusion_matrix.png'

if os.path.exists(results_png):
    print("--- Loss & Performance Curves ---")
    display(Image(filename=results_png))

if os.path.exists(confusion_matrix_png):
    print("--- Confusion Matrix ---")
    display(Image(filename=confusion_matrix_png))

## Step 11: Summary Table of Precision, Recall, and mAP

In [ ]:
import pandas as pd

results_csv = 'runs/train/safecityai_yolov5_run/results.csv'
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    latest_epoch = df.iloc[-1]
    print("--- Final Training Metrics Summary ---")
    metrics_summary = {
        'Epoch': int(latest_epoch['epoch']),
        'Precision': round(float(latest_epoch['metrics/precision']), 4),
        'Recall': round(float(latest_epoch['metrics/recall']), 4),
        'mAP@0.5': round(float(latest_epoch['metrics/mAP_0.5']), 4),
        'mAP@0.5:0.95': round(float(latest_epoch['metrics/mAP_0.5:0.95']), 4)
    }
    for k, v in metrics_summary.items():
        print(f"{k:<15}: {v}")
else:
    print("results.csv not found.")

## Step 12: Run Inference on Unseen Test Set Images

In [ ]:
!python detect.py \
    --weights runs/train/safecityai_yolov5_run/weights/best.pt \
    --img 640 \
    --conf 0.4 \
    --source {DATASET_ROOT}/test/images \
    --name test_image_results

## Step 13: Run Inference on Sample Test Video

In [ ]:
TEST_VIDEO_PATH = '/content/traffic_test.mp4'
if os.path.exists(TEST_VIDEO_PATH):
    !python detect.py \
        --weights runs/train/safecityai_yolov5_run/weights/best.pt \
        --img 640 \
        --conf 0.4 \
        --source {TEST_VIDEO_PATH} \
        --name test_video_results
else:
    print(f"Upload a test video to {TEST_VIDEO_PATH} to run video inference in Colab.")

## Step 14: Export Trained `best.pt` & Results to Google Drive

In [ ]:
import shutil

DRIVE_EXPORT_DIR = '/content/drive/MyDrive/SafeCityAI_Model_Export'
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

if os.path.exists(BEST_WEIGHTS_PATH):
    shutil.copy(BEST_WEIGHTS_PATH, os.path.join(DRIVE_EXPORT_DIR, 'best.pt'))
    print(f"Saved trained model weights to: {DRIVE_EXPORT_DIR}/best.pt")

if os.path.exists(results_csv):
    shutil.copy(results_csv, os.path.join(DRIVE_EXPORT_DIR, 'results.csv'))
    print(f"Saved training results log to: {DRIVE_EXPORT_DIR}/results.csv")